In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns


In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res

def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
window_size = 3
window_size2 = 36


df = pd.read_excel('../2025_08_19_DataMerge/merged_corrected.xlsx')
paths = list(df['path_name'].value_counts().index.values)
for path in paths:
    df.loc[df['path_name']==path, "path_ma"] = df.loc[df['path_name']==path, 'd13c'].rolling(window_size, center=True).mean()
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted['d13c_d18o'] = df_sorted['d13c']/df_sorted['d18o']
df_sorted['age_ws'] = df_sorted['age_absolute'].rolling(window_size, center=True).mean()
df_sorted['age_ws2'] = df_sorted['age_absolute'].rolling(window_size2, center=True).mean()

df_sorted[['d13c_ma','d18o_ma','d13c_d18o_ma']] = df_sorted[['d13c','d18o','d13c_d18o']].rolling(window_size, center=True).mean()
df_sorted[['d13c_ma2','d18o_ma2','d13c_d18o_ma2']] = df_sorted[['d13c','d18o','d13c_d18o']].rolling(window_size2, center=True).mean()
df_sorted[['d13c_std','d18o_std','d13c_d18o_std']] = df_sorted[['d13c','d18o','d13c_d18o']].rolling(window_size2, center=True).std()


#-----------------------RESAMPLE MEAN & STD-----------------------------------------------------

df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_absolute'])
df_resampled_ = df_sorted.set_index('age_dt')
df_resampled_.drop(['Name', 'Mass', 'Amp 44', 'Area 44', 'path_name',
       'path_num', 'path_ma', 'age_ws', 'age_ws2',
       'd13c_ma', 'd18o_ma', 'd13c_d18o_ma', 'd13c_ma2', 'd18o_ma2',
       'd13c_d18o_ma2', 'd13c_std', 'd18o_std', 'd13c_d18o_std'], axis=1, inplace=True)
df_resampled = df_resampled_.resample('30D').mean()

df_resampled.interpolate(inplace=True)
df_resampled[['d13c_ma','d18o_ma', 'd13c_d18o_ma']] = df_resampled[['d13c','d18o', 'd13c_d18o']].rolling(window_size, center=True).mean()
df_resampled[['d13c_ma2','d18o_ma2', 'd13c_d18o_ma2']] = df_resampled[['d13c','d18o','d13c_d18o']].rolling(window_size2, center=True).mean()
df_resampled[['d13c_std','d18o_std','d13c_d18o_std']] = df_resampled[['d13c','d18o','d13c_d18o']].rolling(window_size2, center=True).std()

#-----------------------------------------------------------------------------------------------


#-----------------------CENTRAL DIFF METHOD-----------------------------------------------------
window = 12
col = 'd18o'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])

window = 12
col = 'd13c'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])

window = 12
col = 'd13c_d18o'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])
#-----------------------------------------------------------------------------------------------


#-----------------------------Moving OLS---------------------------------------
coefs_centered = pd.DataFrame()

# Define multiple windows and target columns
windows = [36, 36, 36]
cols = ['d18o', 'd13c','d13c_d18o']

for window, col in zip(windows, cols):
    
    # Prepare data (assumes df_resampled contains all variables)
    data = df_resampled[[col, 'age_absolute']].dropna()

    if len(data) < window:
        print(f"Not enough data for column '{col}' with window {window}. Skipping.")
        continue
    
    # Setup endogenous and exogenous variables
    endog = data[col]
    exog = sm.add_constant(data['age_absolute'])

    # Fit rolling OLS
    rols = RollingOLS(endog, exog, window=window)
    rres = rols.fit()

    # Shift result to center and rename columns
    coefs = rres.params.shift(-(window // 2)).dropna()
    coefs = coefs.rename(columns={'const': f'{col}_intercept', 'age_absolute': f'{col}_slope'})

    # Merge into result
    coefs_centered = coefs_centered.join(coefs, how='outer')

# Add shared age column (assumes time index is same)
coefs_centered['age_absolute'] = df_resampled['age_absolute']

# Drop rows with NaNs in all slope columns (optional)
slope_cols = [f"{col}_slope" for col in cols]
coefs_centered.dropna(subset=slope_cols, how='all', inplace=True)

#-----------------------------------------------------------------------------------------------


#---------------------------------------FFT-----------------------------------------------------
d13c_fft_freq, d13c_fft = FFT_res(df_resampled['d13c'], 30)
d18o_fft_freq, d18o_fft = FFT_res(df_resampled['d18o'], 30)
d18o_fft_freq, d18o_fft = FFT_res(df_resampled['d13c_d18o'], 30)


df_resampled[['d13c_std','d18o_std','d13c_d18o_std']] = df_resampled[['d13c','d18o','d13c_d18o']].rolling(window=window_size2, center=True).std()
d13c_stds = df_resampled.dropna()['d13c_std']
d18o_stds = df_resampled.dropna()['d18o_std']
d13c_std_fft_freq, d13c_std_fft = FFT_res(d13c_stds, 30)
d18o_std_fft_freq, d18o_std_fft = FFT_res(d18o_stds, 30)
#-----------------------------------------------------------------------------------------------


df_resampled



In [ ]:
df


In [ ]:
fred = pd.read_excel('../2025_08_22_FredsData-post2008/FredData_editedMemo.xlsx').dropna()
fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['d13c_ma','d18o_ma']] = fred[['d13c','d18o_pdb']].rolling(window_size, center=True).mean()
fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
fred_resampled = fred.drop(fred.columns[3:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['decimal_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred_resampled

In [ ]:
tide = pd.read_csv('memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)


enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
#fig.add_trace(go.Scatter(x=df['age_absolute'], y=df['d13c'],  mode="markers", marker_color="lightgreen",  text=df['Name'],  name='d13C'),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#120C45', name='Tide Gauge'), secondary_y=True)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)





#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
fig.write_html('d13c_tidegauge.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df['age_absolute'], y=df['d13c'],  mode="markers", marker_color="lightgreen",  text=df['Name'],  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#120C45', name='Tide Gauge'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)





#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
fig.write_html('d13c_tidegauge.html')


In [ ]:
fred

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_d18o'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_d18o_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_d18o_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_d18o_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#120C45', name='Tide Gauge'), secondary_y=True)


fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)

#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
#fig.write_html('d13c_tidegauge.html')
